# DUET Classification Notebook

Этот блокнот приведён в соответствие с текущей архитектурой DUET для **классификации временных рядов**:
- **TCM** сегментирует ряд на патчи, выделяет тренд/сезонность и маршрутизирует их по temporal-кластерам.
- **CCM** работает в частотной области, учит межканальные связи и разреживает маску.
- **Fusion** объединяет temporal-признаки и маску каналов, после чего **classification head** выдаёт logits классов.

Ниже используются параметры `K_t`, `K_c`, `d_c`, `top_k`, а также RevIN/InstanceNorm в соответствии с `architecture.md`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import sys
sys.path.append("/content/drive/MyDrive/stocks/duet_class")

from pipeline.config import DUETConfig
from pipeline import preprocess, train, evaluate, predict
from pipeline.prepare_and_check import FinancialTimeSeriesPreparer
from pipeline.wf_slicer import GlobalNormConfig, SplitConfig, WalkForwardWindowSlicerVec, WindowConfig
from duet.model import DUETModel
from torch.utils.data import DataLoader, TensorDataset
import torch
import joblib
import random

import numpy as np
from sklearn.utils.class_weight import compute_class_weight


In [ ]:
# Настройки
joblib_path = "/content/drive/MyDrive/stocks/Data/NEARUSDT/nearusdt_ob_indicators_TF_5_metric_v2-long_rsi_labels-1.joblib"
device = "cuda" if torch.cuda.is_available() else "cpu"

config = DUETConfig(
    # =========================
    # Общие параметры
    # =========================
    timestamp_col = "timestamp",
    features = [
       'Open', 'High', 'Low', 'Close',
       'Volume',
      #  'sigma_bollinger_7','sigma_bollinger_14',
      #  'sigma_log_return_7',  'sigma_true_range_7', 'min_extr_distance_7',
      #  'max_extr_distance_7', 'efficiency_ratio_7',
      #  'sigma_log_return_14',  'sigma_true_range_14', 'min_extr_distance_14',
      #  'max_extr_distance_14', 'efficiency_ratio_14'
       'up_bb','low_bb','RSI_Close',
      #  'Depth_ratio_1','Depth_ratio_3','Depth_ratio_5','Depth_ratio_5','Depth_ratio_15',
        # 'Depth_ask_1','Depth_bid_1','Depth_ask_3','Depth_bid_3','Depth_ask_5','Depth_bid_5','Depth_ask_8','Depth_bid_8','Depth_ask_15','Depth_bid_15',
      #  'RSI_Depth_ask_1','RSI_Depth_ask_3','RSI_Depth_ask_5','RSI_Depth_ask_8','RSI_Depth_bid_1','RSI_Depth_bid_3','RSI_Depth_bid_5','RSI_Depth_bid_8',
      #  'Depth_ratio_30_k','Depth_ratio_15_k',
       'Depth_ratio_8_k','Depth_ratio_5_k','Depth_ratio_3_k',
       'Total_depth_1','Total_depth_3','Total_depth_5','Total_depth_8','Total_depth_15',
       'Impulse_1','Impulse_3','Impulse_5','Impulse_8','Impulse_15',
      #  'Velocity_1','Velocity_3','Velocity_5','Velocity_8','Velocity_15',
      #  'count_long_short_ratio', 'sum_taker_long_short_vol_ratio',
      #  'Depth_imbalance_1','Depth_imbalance_3','Depth_imbalance_5','Depth_imbalance_8','Depth_imbalance_15',
    ],                           # Название колонок для обучения
    forecast = 'Labels2:',          # Название колонки с таргетом
    not_to_normalise = [],       # Название колонок, которые НЕ НАДО нормализовать
    scaler = 'MINMAX',         # Тип нормализации (STD, MINMAX, QUANT)
    predict_type = 'detect',     # Детекция "detect" или предикт "next" следующей свечи
    seq_len = 48,                # Длина входной последовательности
    num_classes = 3,             # кол-во классов
    patch_len = 24,               # Длина патча (TCM)
    stride = 12,                  # Шаг между патчами (TCM)
    moving_avg = 25,              # Размер окна скользящего сглаживания

    K_t = 2,                    # Кол-во временных кластеров (TCM)
    K_c = 4,                    # Кол-во кластеров каналов (CCM)
    d_c = 32,                   # Размерность embedding каналов (CCM)
    top_k = 2,                  # Кол-во связей при разреживании маски
    use_revin = True,           # Включить RevIN/InstanceNorm
    revin_affine = True,        # Использовать affine параметры в RevIN
    revin_eps = 1e-5,           # Эпсилон для стабильности RevIN

    # =========================
    # Параметры модели
    # =========================
    d_model = 64,               # Размерность скрытого пространства в attention
    d_ff = 256,                 # Размерность feedforward слоя
    n_heads = 4,                # Количество голов в multi-head attention
    e_layers = 2,               # Количество слоев в encoder (CCM)
    dropout = 0.2,            # Dropout во всех слоях attention
    fc_dropout = 0.2,         # Dropout в выходном head слое
    activation = "relu",        # Активационная функция (relu, gelu, elu)
    num_experts = 4,            # Число экспертов (в Router, если используется)
    report_freq = 10,            # Частота появления confusion matrix

    # =========================
    # Режимы обработки
    # =========================
    CI = True,                 # Channel-Independent режим (если False — shared weights)
    use_router = True,          # Включить распределительный роутер
    timeenc = 1,                # Использовать time encoding (0 = без, 1 = sin/cos и т.п.)

    # =========================
    # Настройки обучения
    # =========================
    batch_size=32,          # можно немного увеличить при хорошем GPU
    epochs=300,             # больше эпох для более тонкой настройки
    learning_rate=5e-4,     # стандартный LR для Adam
    weight_decay=1e-5,
    patience=300,            # early stopping не слишком строгий

    # =========================
    # Прочее
    # =========================
    checkpoint_best = '/content/drive/MyDrive/stocks/duet_class/weights/best_val_acc_weights_001.pt',       # Путь для сохранения лучших по val_accuracy весов
    checkpoint_final = '/content/drive/MyDrive/stocks/duet_class/weights/final_weights_001.pt',      # Путь для сохранения финальных весов
    seed = 11,                  # Фиксированное зерно генератора случайных чисел
    verbose = True,            # Печать хода обучения
)

"""
Устанавливает seed для numpy, random, torch (вкл. CUDA).
Гарантирует воспроизводимость.
"""

random.seed(config.seed)
np.random.seed(config.seed)
torch.manual_seed(config.seed)
torch.cuda.manual_seed_all(config.seed)

## Варианты запуска обучения (наборы конфигураций)

Ниже — типовые варианты запуска, которые отражают логику обучения DUET: end-to-end обучение **TCM → CCM → Fusion → Head** с кросс-энтропией. Для каждого варианта меняются только ключевые параметры архитектуры и кластеризации, остальные поля берутся из базовой конфигурации.

**1) Базовый режим (баланс точности и стабильности):**
```python
DUETConfig(
    K_t=2, K_c=4, d_c=32, top_k=2,
    use_router=True, use_revin=True,
    d_model=64, d_ff=128, e_layers=2,
)
```

**2) Больше временных режимов (сложная динамика):**
```python
DUETConfig(
    K_t=4, patch_len=12, stride=6,
    use_router=True, num_experts=6,
    d_model=96, d_ff=256,
)
```

**3) Усиленная кластеризация каналов (много каналов/шумные связи):**
```python
DUETConfig(
    K_c=6, d_c=48, top_k=3,
    e_layers=3, dropout=0.15,
)
```

**4) Ablation без temporal-router (проверка вклада маршрутизации):**
```python
DUETConfig(
    use_router=False, K_t=1,
    d_model=64, d_ff=128,
)
```

Для запуска достаточно заменить поля в блоке `config = DUETConfig(...)` ниже или создать отдельные конфиги и прогнать обучение в цикле.


In [ ]:
df = joblib.load(joblib_path)
df['Time_close'] = df['Time_close'] / 1000
df = df.rename(columns={"Time_close": "timestamp"})
df.columns

In [ ]:
import pandas as pd

def efficiency_ratio(series: pd.Series, window: int) -> pd.Series:
    # Направленное изменение
    direction = series.diff(window).abs()

    # Шум: сумма абсолютных изменений внутри окна
    volatility = series.diff().abs().rolling(window=window).sum()

    # ER = направленное / суммарное
    er = direction / volatility

    return er

df['er_15'] = efficiency_ratio(df['Close'], window=15)
df['er_30'] = efficiency_ratio(df['Close'], window=30)
# df['er_60'] = efficiency_ratio(df['Close'], window=60)
# df['er_90'] = efficiency_ratio(df['Close'], window=90)

In [ ]:
# --- 1. Предобработка данных ---
# Загрузка данных
# df = joblib.load(joblib_path)
preparer = FinancialTimeSeriesPreparer(
    tz="UTC",
    timestamp_col="timestamp",
    drop_warmup=True,
)
df, _ = preparer.prepare(df, ensure_ohlcv=True)

split = SplitConfig(
    n_folds=1,
    mode="expanding",
    ratios=(0.75, 0.25, 0.0),
    step_size=None,
    gap=0,
    sliding_train_size=None,
)

if config.predict_type.upper() == "DETECT":
    y_end_offset = 0
elif config.predict_type.upper() == "NEXT":
    y_end_offset = 1
else:
    raise ValueError("config.predict_type должен быть 'DETECT' или 'NEXT'")

window = WindowConfig(
    x_window=config.seq_len,
    x_end_offset=0,
    y_window=1,
    y_end_offset=y_end_offset,
    allow_left_context_for_x=False,
)

scaler_map = {"STD": "standard", "MINMAX": "minmax", "QUANT": "quantile", "NONE": "none"}
global_norm = GlobalNormConfig(scaler=scaler_map.get(config.scaler.upper(), "none"))

slicer = WalkForwardWindowSlicerVec(
    split=split,
    window=window,
    global_norm=global_norm,
    no_norm_cols=config.not_to_normalise,
    eps=1e-12,
    drop_incomplete_last_fold=True,
)

out = slicer.split_and_window(
    X=df[config.features],
    y=df[[config.forecast]],
)
fold0 = out["fold_0"]

x_train = fold0["train"]["X"]
y_train = fold0["train"]["y"][:, 0, 0].astype(int)
x_val = fold0["val"]["X"]
y_val = fold0["val"]["y"][:, 0, 0].astype(int)


# Балансируем (если не нужно, то закомментировать)
x_train, y_train = preprocess.balance_windows(x_train, y_train)
x_val, y_val = preprocess.balance_windows(x_val, y_val)

# Проверяем балансировку
unique, counts = np.unique(y_train, return_counts=True)
print("Классы после балансировки:", dict(zip(unique, counts)))

# --- 3. Подсчет весов классов ---
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
print(f"Распределение весов между класcами {class_weights}")

# --- 4. Создание DataLoader'ов ---
train_dataset = TensorDataset(
    torch.tensor(x_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long)
)
val_dataset = TensorDataset(
    torch.tensor(x_val, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.long)
)

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config.batch_size)

# --- 6. Инициализация модели ---
model = DUETModel(config).to(device)

# --- 7. Обучение модели ---
trained_model = train.train_model(
    model,
    config,
    train_loader,
    val_loader,
    device=device,
    class_weights = class_weights
)


In [ ]:
# Оценка модели
y_true, y_pred = evaluate.evaluate_model(model, val_loader, device="cuda")

# Подсчёт метрик
metrics = evaluate.compute_metrics(y_true, y_pred)
print("\nMetrics:")
for metric, value in metrics.items():
    print(f"{metric.upper()}: {value:.2f}")


# Матрица ошибок
evaluate.plot_confusion_matrix(y_true, y_pred)

# Примеры предсказаний
evaluate.plot_classification_examples(y_true, y_pred, n=10)

In [ ]:
# Загрузка весов

model.load_state_dict(torch.load(config.checkpoint_best)) # загрузка лучших весов
# model.load_state_dict(torch.load(config.checkpoint_final)) # загрузка финальных весов
model.eval()

In [ ]:
from pipeline.visualiser import plot_classification_forecast, plot_roc_auc, plot_pr_auc
from pipeline.predict import predict_dataset_batched

# Предсказания
df_val['pred_pivots'], y_probs = predict_dataset_batched(model, df_val, config, device="cuda")
# print(df_val)
# Выбор валидных индексов, где были предсказания
valid_mask = ~df_val['pred_pivots'].isna()

# Выравнивание по индексу
y_true = df_val.loc[valid_mask, config.forecast].values.astype(int)
y_pred_proba = y_probs[valid_mask.values]  # valid_mask должен быть ndarray такой же длины

# ROC-AUC
plot_roc_auc(y_true, y_pred_proba, n_classes=config.num_classes)

# PR-AUC
plot_pr_auc(y_true, y_pred_proba, n_classes=config.num_classes)

# Диагностика предсказаний
df_val.dropna(inplace=True)
print(df_val['pred_pivots'].value_counts(dropna=False))


In [ ]:
plot_classification_forecast(
    df_val,
    price_col="Close",
    target_col="pivots",
    forecast_col="pred_pivots",
    class_labels=["Valley", "Neutral", "Peak"],
    start=500,
    length=300,
    title="Classification Forecast: Price and Predicted Regimes"
)